In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from NOTEBOOKS.calculateEVS import *
from MODELS.pipeline import *
from MODELS.teamInfo import mainStartingFive, teamStarPlayer, projectedStartingFive

### Load Model

In [2]:
model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MODEL.pkl')
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s25= pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
df = pd.concat([s25, s26])

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

/var/folders/nw/9w4r5hrd05s122kt5hg_t8bw0000gn/T/ipykernel_75942/769474497.py:6: DtypeWarning: Columns (32,33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


### Update projected starting lineups

In [16]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor/MODELS/teamInfo.py
Updated 16 teams with confirmed lineups


### Top EVs for single bets

In [17]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['ODDS'] <= 200) & (usData['ODDS'] >= -200)]

results = calculateSingleBets(df, singlePTSBookies, model, features, current_date, edge_threshold=0.20, stake=10, 
                     variance_inflation=1.1, distribution_type='t', stat_col='PTS', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

singleBets = results
singleBets = singleBets[(singleBets['SIGMA FLAG'] == 'Med') | (singleBets['SIGMA FLAG'] == 'Low')].sort_values(by='EV%', ascending=False).reset_index(drop=True)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets_{today}.csv', index=False)
singleBets.head()

Processing single bets with single model...


,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,RECOMMENDATION,OVER%,UNDER%,IMPLIED PROB,MODEL PROB,EDGE,EV%,KELLY_FRACTION,KELLY_DOLLARS,CONFIDENCE INTERVAL,INTERVAL WIDTH,SIGMA,SIGMA FLAG,EXPECTED ROI,SIMULATION_METHOD
0,Nicolas Batum,BetMGM,player_points,3.5,110,Over,8.78,1,0.877,0.123,0.476,0.877,0.401,8.42,0.766,2.75,"(0.0, 18.9)",18.92,5.18,Med,0.84,Monte Carlo
1,Mitchell Robinson,BetMGM,player_points,4.5,-110,Over,10.74,1,0.898,0.102,0.524,0.898,0.375,7.15,0.787,2.27,"(0.0, 21.6)",21.56,5.52,Med,0.72,Monte Carlo
2,Mitchell Robinson,FanDuel,player_points,4.5,-111,Over,10.74,1,0.898,0.102,0.526,0.898,0.372,7.08,0.786,2.25,"(0.0, 21.6)",21.56,5.52,Med,0.71,Monte Carlo
3,Andre Drummond,DraftKings,player_points,4.5,104,Over,9.30,1,0.836,0.164,0.490,0.836,0.346,7.06,0.679,2.60,"(0.0, 20.5)",20.51,5.72,Med,0.71,Monte Carlo
4,Mitchell Robinson,DraftKings,player_points,4.5,-112,Over,10.74,1,0.898,0.102,0.528,0.898,0.370,7.01,0.785,2.23,"(0.0, 21.6)",21.56,5.52,Med,0.70,Monte Carlo


## Top EVs for 2 leg bets

### Underdog picks

In [18]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=100, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogPairs = results
underdogPairs = underdogPairs[
    underdogPairs[['sigma_flag1', 'sigma_flag2']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs_{today}.csv', index=False)
underdogPairs.head()

,player1,player2,line1,line2,pred1,pred2,model_side1,model_side2,prob1,prob2,prob_both,edge1,edge2,combined_edge,ev_percent,kelly_full,recommendation,confidence_interval1,confidence_interval2,interval_width1,interval_width2,sigma1,sigma2,sigma_flag1,sigma_flag2,simulation_method
0,Quenton Jackson,Andre Drummond,6.5,3.5,11.93,9.30,over,over,0.866,0.876,0.6830,0.288,0.298,0.293,1.05,0.524,0,"(0.9, 23.0)","(0.0, 20.5)",22.06,20.51,5.63,5.72,Med,Med,Monte Carlo
1,Keaton Wallace,Andre Drummond,4.5,3.5,8.36,9.30,over,over,0.807,0.876,0.6365,0.229,0.298,0.264,0.91,0.455,0,"(0.0, 18.6)","(0.0, 20.5)",18.57,20.51,5.21,5.72,Med,Med,Monte Carlo
2,Keaton Wallace,Quenton Jackson,4.5,6.5,8.36,11.93,over,over,0.807,0.866,0.6289,0.229,0.288,0.258,0.89,0.443,0,"(0.0, 18.6)","(0.9, 23.0)",18.57,22.06,5.21,5.63,Med,Med,Monte Carlo
3,Andre Drummond,Aaron Gordon,3.5,17.5,9.30,15.86,over,under,0.876,0.678,0.5349,0.298,0.100,0.199,0.60,0.302,0,"(0.0, 20.5)","(7.2, 24.5)",20.51,17.33,5.72,4.42,Med,Low,Monte Carlo
4,Quenton Jackson,Aaron Gordon,6.5,17.5,11.93,15.86,over,under,0.866,0.678,0.5285,0.288,0.100,0.194,0.59,0.293,0,"(0.9, 23.0)","(7.2, 24.5)",22.06,17.33,5.63,4.42,Med,Low,Monte Carlo


### Prizepicks picks

In [19]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=100, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)
pairsPrizepicks = results
pairsPrizepicks = pairsPrizepicks[
    pairsPrizepicks[['sigma_flag1', 'sigma_flag2']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs_{today}.csv', index=False)
pairsPrizepicks.head()

,player1,player2,line1,line2,pred1,pred2,model_side1,model_side2,prob1,prob2,prob_both,edge1,edge2,combined_edge,ev_percent,kelly_full,recommendation,confidence_interval1,confidence_interval2,interval_width1,interval_width2,sigma1,sigma2,sigma_flag1,sigma_flag2,simulation_method
0,Quenton Jackson,John Konchar,6.5,3.5,11.93,8.52,over,over,0.866,0.887,0.6912,0.288,0.309,0.298,1.07,0.537,0,"(0.9, 23.0)","(0.0, 17.7)",22.06,17.72,5.63,4.69,Med,Low,Monte Carlo
1,Keaton Wallace,John Konchar,4.5,3.5,8.36,8.52,over,over,0.807,0.887,0.6442,0.229,0.309,0.269,0.93,0.466,0,"(0.0, 18.6)","(0.0, 17.7)",18.57,17.72,5.21,4.69,Med,Low,Monte Carlo
2,John Konchar,Blake Wesley,3.5,6.0,8.52,9.92,over,over,0.887,0.802,0.6406,0.309,0.224,0.267,0.92,0.461,0,"(0.0, 17.7)","(0.0, 20.6)",17.72,20.56,4.69,5.43,Low,Med,Monte Carlo
3,Quenton Jackson,Keaton Wallace,6.5,4.5,11.93,8.36,over,over,0.866,0.807,0.6289,0.288,0.229,0.258,0.89,0.443,0,"(0.9, 23.0)","(0.0, 18.6)",22.06,18.57,5.63,5.21,Med,Med,Monte Carlo
4,Quenton Jackson,Blake Wesley,6.5,6.0,11.93,9.92,over,over,0.866,0.802,0.6254,0.288,0.224,0.256,0.88,0.438,0,"(0.9, 23.0)","(0.0, 20.6)",22.06,20.56,5.63,5.43,Med,Med,Monte Carlo


## 3 leg parlay

### Underdog picks

In [20]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.10, stake=100, 
                     variance_inflation=1.1, distribution_type='t', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogTrios = threeLeg[
    threeLeg[['sigma_flag1', 'sigma_flag2', 'sigma_flag3']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios_{today}.csv', index=False)
underdogTrios.head()

,player1,player2,player3,line1,line2,line3,pred1,pred2,pred3,model_side1,model_side2,model_side3,prob1,prob2,prob3,prob_all_three,edge1,edge2,edge3,combined_edge,ev_percent,kelly_full,recommendation,confidence_interval1,confidence_interval2,confidence_interval3,interval_width1,interval_width2,interval_width3,sigma1,sigma2,sigma3,sigma_flag1,sigma_flag2,sigma_flag3,simulation_method
0,Keaton Wallace,Quenton Jackson,Andre Drummond,4.5,6.5,3.5,8.36,11.93,9.30,over,over,over,0.807,0.866,0.876,0.4961,0.229,0.288,0.298,0.272,1.98,0.395,0,"(0.0, 18.6)","(0.9, 23.0)","(0.0, 20.5)",18.57,22.06,20.51,5.21,5.63,5.72,Med,Med,Med,Monte Carlo
1,Quenton Jackson,Andre Drummond,Aaron Gordon,6.5,3.5,17.5,11.93,9.30,15.86,over,over,under,0.866,0.876,0.678,0.4168,0.288,0.298,0.100,0.229,1.50,0.300,0,"(0.9, 23.0)","(0.0, 20.5)","(7.2, 24.5)",22.06,20.51,17.33,5.63,5.72,4.42,Med,Med,Low,Monte Carlo
2,Keaton Wallace,Andre Drummond,Aaron Gordon,4.5,3.5,17.5,8.36,9.30,15.86,over,over,under,0.807,0.876,0.678,0.3885,0.229,0.298,0.100,0.209,1.33,0.266,0,"(0.0, 18.6)","(0.0, 20.5)","(7.2, 24.5)",18.57,20.51,17.33,5.21,5.72,4.42,Med,Med,Low,Monte Carlo
3,Keaton Wallace,Quenton Jackson,Aaron Gordon,4.5,6.5,17.5,8.36,11.93,15.86,over,over,under,0.807,0.866,0.678,0.3838,0.229,0.288,0.100,0.206,1.30,0.261,0,"(0.0, 18.6)","(0.9, 23.0)","(7.2, 24.5)",18.57,22.06,17.33,5.21,5.63,4.42,Med,Med,Low,Monte Carlo


### Prizepicks picks

In [21]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.40, stake=100, 
                     variance_inflation=1.1, distribution_type='normal', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

triosPrizepicks = threeLeg[
    threeLeg[['sigma_flag1', 'sigma_flag2', 'sigma_flag3']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='ev_percent', ascending=False).reset_index(drop=True)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios_{today}.csv', index=False)
triosPrizepicks.head()

,player1,player2,player3,line1,line2,line3,pred1,pred2,pred3,model_side1,model_side2,model_side3,prob1,prob2,prob3,prob_all_three,edge1,edge2,edge3,combined_edge,ev_percent,kelly_full,recommendation,confidence_interval1,confidence_interval2,confidence_interval3,interval_width1,interval_width2,interval_width3,sigma1,sigma2,sigma3,sigma_flag1,sigma_flag2,sigma_flag3,simulation_method
0,Quenton Jackson,Keaton Wallace,John Konchar,6.5,4.5,3.5,11.93,8.36,8.52,over,over,over,0.830,0.776,0.852,0.4443,0.252,0.198,0.274,0.241,1.67,0.333,0,"(0.9, 23.0)","(0.0, 18.6)","(0.0, 17.7)",22.06,18.57,17.72,5.63,5.21,4.69,Med,Med,Low,Monte Carlo
1,Quenton Jackson,John Konchar,Blake Wesley,6.5,3.5,6.0,11.93,8.52,9.92,over,over,over,0.830,0.863,0.756,0.4388,0.252,0.285,0.178,0.238,1.63,0.327,0,"(0.9, 23.0)","(0.0, 17.7)","(0.0, 20.6)",22.06,17.72,20.56,5.63,4.69,5.43,Med,Low,Med,Monte Carlo
2,Quenton Jackson,Collin Murray-Boyles,John Konchar,6.5,10.5,3.5,11.93,13.41,8.52,over,over,over,0.830,0.710,0.852,0.4064,0.252,0.132,0.274,0.219,1.44,0.288,0,"(0.9, 23.0)","(3.0, 23.9)","(0.0, 17.7)",22.06,20.89,17.72,5.63,5.33,4.69,Med,Med,Low,Monte Carlo
3,Keaton Wallace,John Konchar,Blake Wesley,4.5,3.5,6.0,8.36,8.52,9.92,over,over,over,0.768,0.863,0.756,0.4064,0.190,0.285,0.178,0.218,1.44,0.288,0,"(0.0, 18.6)","(0.0, 17.7)","(0.0, 20.6)",18.57,17.72,20.56,5.21,4.69,5.43,Med,Low,Med,Monte Carlo
4,Quenton Jackson,Keaton Wallace,Blake Wesley,6.5,4.5,6.0,11.93,8.36,9.92,over,over,over,0.830,0.776,0.756,0.3945,0.252,0.198,0.178,0.209,1.37,0.273,0,"(0.9, 23.0)","(0.0, 18.6)","(0.0, 20.6)",22.06,18.57,20.56,5.63,5.21,5.43,Med,Med,Med,Monte Carlo


In [9]:
def count_line_hits(player_df, line, category, game_windows=[5, 10, 15]):
    results = {}
    player_df_sorted = player_df.sort_values('GAME_DATE')
    total_games = len(player_df_sorted)

    for window in game_windows:
        # Handle players with fewer games
        if total_games < window:
            last_n_games = player_df_sorted
        else:
            last_n_games = player_df_sorted.tail(window)

        if category == 'player_points':
            hits = (last_n_games['PTS'] > line).sum()
        elif category == 'player_assists':
            hits = (last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds':
            hits = (last_n_games['REB'] > line).sum()
        elif category == 'player_threes':
            hits = (last_n_games['FG3M'] > line).sum()
        elif category == 'player_blocks':
            hits = (last_n_games['BLK'] > line).sum()
        elif category == 'player_steals':
            hits = (last_n_games['STL'] > line).sum()
        elif category == 'player_field_goals':
            hits = (last_n_games['FGM'] > line).sum()
        elif category == 'player_frees_made':
            hits = (last_n_games['FTM'] > line).sum()
        elif category == 'player_points_rebounds_assists':
            hits = (last_n_games['PTS'] + last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_points_rebounds':
            hits = (last_n_games['PTS'] + last_n_games['REB'] > line).sum()
        elif category == 'player_points_assists':
            hits = (last_n_games['PTS'] + last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds_assists':
            hits = (last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_turnovers':
            hits = (last_n_games['TOV'] > line).sum()
        else:
            hits = 0

        results['NAME'] = player_df_sorted['PLAYER_NAME'].iloc[0] if total_games > 0 else 'Unknown'
        results['CATEGORY'] = category
        results['LINE'] = line
        results[f'LAST {window}'] = hits
        results[f'HIT RATE % LAST {window}'] = round(hits / window, 2)


    return results

prizePicks = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks')]
prizePicks= prizePicks.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
prizePicks

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
160,PrizePicks,player_points,Pascal Siakam,Over,26.5,-137,2025-10-31,2025-10-31T21:56:50Z
162,PrizePicks,player_points,Jalen Johnson,Over,20.5,-137,2025-10-31,2025-10-31T21:56:50Z
164,PrizePicks,player_points,Kristaps Porzingis,Over,18.5,-137,2025-10-31,2025-10-31T21:56:50Z
166,PrizePicks,player_points,Nickeil Alexander-Walker,Over,16.5,-137,2025-10-31,2025-10-31T21:56:50Z
168,PrizePicks,player_points,Aaron Nesmith,Over,15.5,-137,2025-10-31,2025-10-31T21:56:50Z
...,...,...,...,...,...,...,...,...
2408,PrizePicks,player_blocks_steals,Brice Sensabaugh,Over,0.5,-137,2025-11-01,2025-10-31T21:57:17Z
2410,PrizePicks,player_blocks_steals,Brook Lopez,Over,1.5,-137,2025-11-01,2025-10-31T21:57:08Z
2412,PrizePicks,player_blocks_steals,Yves Missi,Over,1.5,-137,2025-11-01,2025-10-31T21:57:08Z
2414,PrizePicks,player_blocks_steals,Herb Jones,Over,1.5,-137,2025-11-01,2025-10-31T21:57:08Z


In [ ]:
line_hit_data = []

for index, row in prizePicks.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = df[df['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

        
        

Saved 106 records for player_points to player_points.csv
Saved 65 records for player_rebounds to player_rebounds.csv
Saved 43 records for player_assists to player_assists.csv
Saved 19 records for player_threes to player_threes.csv
Saved 4 records for player_blocks to player_blocks.csv
Saved 12 records for player_steals to player_steals.csv
Saved 36 records for player_field_goals to player_field_goals.csv
Saved 32 records for player_frees_made to player_frees_made.csv
Saved 25 records for player_frees_attempts to player_frees_attempts.csv
Saved 101 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 95 records for player_points_rebounds to player_points_rebounds.csv
Saved 91 records for player_points_assists to player_points_assists.csv
Saved 66 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 15 records for player_turnovers to player_turnovers.csv
Saved 22 records for player_blocks_steals to player_blocks_steals.csv

All catego

In [11]:
underdog = dfsData[(dfsData['BOOKMAKER'] == 'Underdog')]
underdog= underdog.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
line_hit_data = []

for index, row in underdog.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = df[df['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

Saved 70 records for player_points to player_points.csv
Saved 19 records for player_rebounds to player_rebounds.csv
Saved 16 records for player_assists to player_assists.csv
Saved 15 records for player_threes to player_threes.csv
Saved 2 records for player_steals to player_steals.csv
Saved 4 records for player_frees_made to player_frees_made.csv
Saved 91 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 39 records for player_points_rebounds to player_points_rebounds.csv
Saved 30 records for player_points_assists to player_points_assists.csv
Saved 22 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 6 records for player_turnovers to player_turnovers.csv
Saved 1 records for player_blocks_steals to player_blocks_steals.csv

All category files saved to ../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG
